### Connection with Cosmos DB

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("eventhub-scope1")

In [0]:
CONNECTION_STR = dbutils.secrets.get(
    scope="eventhub-scope1",
    key="CosmosDB"
)

In [0]:
cosmos_base_config = {
    "spark.cosmos.accountEndpoint": "https://banking-cosmos-db.documents.azure.com:443/",
    "spark.cosmos.accountKey": CONNECTION_STR,
    "spark.cosmos.write.strategy": "ItemOverwrite",
    "spark.cosmos.write.bulk.enabled": "true"
}

### 1. Fraud Risk Alert

In [0]:
from pyspark.sql.functions import col
fraud_risk_df = spark.table("bankingdata.businessinsights.fraud_risk") \
    .withColumn("id", col("transaction_id").cast("string"))

In [0]:
fraud_risk_df.write \
    .format("cosmos.oltp") \
    .options(**cosmos_base_config) \
    .option("spark.cosmos.database", "BankingDB") \
    .option("spark.cosmos.container", "Fraud Risk") \
    .mode("APPEND") \
    .save()

### 2. Transaction Velocity Spike


In [0]:
velocity_risk_df = spark.table("bankingdata.businessinsights.velocity_risk")

In [0]:
velocity_risk_df = velocity_risk_df.withColumn(
    "id", col("transaction_id").cast("string")
)

In [0]:
velocity_risk_df.write \
    .format("cosmos.oltp") \
    .options(**cosmos_base_config) \
    .option("spark.cosmos.database", "BankingDB") \
    .option("spark.cosmos.container", "Transaction Velocity") \
    .mode("APPEND") \
    .save()

### 3. Location Anomaly

In [0]:
location_risk_df = spark.table("bankingdata.businessinsights.location_risk")

In [0]:
location_risk_df.write \
    .format("cosmos.oltp") \
    .options(**cosmos_base_config) \
    .option("spark.cosmos.database", "BankingDB") \
    .option("spark.cosmos.container", "Location Anomaly") \
    .mode("APPEND") \
    .save()